# Ingest constructors.json file
1. Read the file using spark dataframe reader API
2. Add Metadata Columns 
   - Source File
   - Ingestion Timestamp
3. Write to bronze delta table  

In [0]:
%run "../00-common/01.environment-config"


In [0]:
%run "../00-common/02.bronze-helpers"

In [0]:
val source_file= landing_folder_path + "/constructors.json"
val table_name= catalog_name + "." + bronze_schema + "." + "constructors"

In [0]:
import org.apache.spark.sql.types.{StructType,StructField,StringType}

val constructors_schema=StructType(Seq(
  StructField("constructorId", StringType),
  StructField("name", StringType),
  StructField("nationality", StringType),
  StructField("url", StringType)
))

val constructors_df=spark.read.format("json")
.option("mode", "FAILFAST")
.schema(constructors_schema)
.load(source_file)

display(constructors_df)

In [0]:

val constructors_final_df= add_ingestion_metadata(constructors_df)

display(constructors_final_df)

#### Step 3 - Write to bronze delta table

In [0]:
constructors_final_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

In [0]:
display(spark.table(table_name))